### Import relevent libraries

In [38]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
#import sqlite3
from sqlalchemy import create_engine
import os

### To make results reproducible

In [39]:
random.seed(42)
np.random.seed(42)

#### Roaming partner Data

In [40]:
PARTNERS = [
    ('Deutsche Telekom',   'Germany',        'Europe',   0.96),
    ('Vodafone UK',        'United Kingdom', 'Europe',   0.94),
    ('Orange France',      'France',         'Europe',   0.95),
    ('Telecom Italia',     'Italy',          'Europe',   0.88),
    ('NTT Docomo',         'Japan',          'APAC',     0.97),
    ('SK Telecom',         'South Korea',    'APAC',     0.96),
    ('SingTel',            'Singapore',      'APAC',     0.93),
    ('Optus',              'Australia',      'APAC',     0.91),
    ('Zain KSA',           'Saudi Arabia',   'MEA',      0.82),
    ('Etisalat UAE',       'UAE',            'MEA',      0.87),
    ('MTN South Africa',   'South Africa',   'MEA',      0.79),
    ('Safaricom',          'Kenya',          'MEA',      0.75),
    ('AT&T',               'USA',            'Americas', 0.95),
    ('T-Mobile USA',       'USA',            'Americas', 0.94),
    ('Claro Brazil',       'Brazil',         'Americas', 0.83),
    ('Telcel Mexico',      'Mexico',         'Americas', 0.80),
    ('Telenor Norway',     'Norway',         'Europe',   0.97),
    ('Turkcell',           'Turkey',         'MEA',      0.84),
    ('Jio India',          'India',          'APAC',     0.92),
    ('China Mobile',       'China',          'APAC',     0.78),
]
TECHNOLOGIES = ['2G', '3G', '4G', 'VoLTE', '5G']
TECH_WEIGHTS  = [0.05, 0.10, 0.45, 0.25, 0.15] # Probabilities for selecting each technology


In [41]:
TEST_TYPES = {
    '2G':    ['Voice MO', 'Voice MT', 'SMS MO', 'SMS MT'],
    '3G':    ['Voice MO', 'Voice MT', 'SMS MO', 'Data'],
    '4G':    ['Voice MO', 'Voice MT', 'SMS MO', 'Data', 'VoLTE MO'],
    'VoLTE': ['VoLTE MO', 'VoLTE MT', 'SMS over IMS'],
    '5G':    ['Voice MO', 'Data', 'VoNR MO', 'SMS'],
} # Types of test for each technology

FAILURE_REASONS = [
    'No answer from remote side',
    'HLR/HSS lookup failure',
    'Routing error - wrong IPX route',
    'CAMEL trigger missing',
    'Diameter timeout',
    'APN not configured',
    'Authentication failure',
    'STP signalling error',
]
# Reasons for the failed test cases

## Genreate 6 months of DATA

In [42]:
START_DATE = datetime(2024, 7, 1)
END_DATE = datetime(2024, 12, 31)

In [43]:
records = []
test_id = 1

date= START_DATE

while date <= END_DATE:
    for partner_name, country, region, base_pass_rate in PARTNERS:
        # Each partner gets 3-8 test cases per day
        num_tests = random.randint(3, 8)
        for _ in range(num_tests):
            tech = random.choices(TECHNOLOGIES, weights=TECH_WEIGHTS)[0]
            test_type = random.choice(TEST_TYPES[tech])
            # Add some weekly degradation for certain partners
            week_num = (date - START_DATE).days // 7
            if partner_name in ['Safaricom', 'China Mobile', 'Claro Brazil']:
                pass_rate = max(0.60, base_pass_rate - (week_num * 0.005))
            elif partner_name in ['NTT Docomo', 'Deutsche Telekom']:
                pass_rate = min(0.99, base_pass_rate + (week_num * 0.002))
            else:
                pass_rate = base_pass_rate + random.uniform(-0.03, 0.03)
            result = 'pass' if random.random() < pass_rate else 'fail'
            failure_reason = ''
            if result == 'fail':
                failure_reason = random.choice(FAILURE_REASONS)
            duration_sec = round(random.uniform(0.5, 45.0), 1) if result=='pass' else None
            records.append({
                'test_id':        test_id,
                'test_date':      date.strftime('%Y-%m-%d'),
                'month':          date.strftime('%Y-%m'),
                'week':           date.strftime('%Y-W%V'),
                'partner_name':   partner_name,
                'country':        country,
                'region':         region,
                'technology':     tech,
                'test_type':      test_type,
                'result':         result,
                'failure_reason': failure_reason,
                'duration_sec':   duration_sec,
            })
            test_id += 1
    date += timedelta(days=1)
 
df = pd.DataFrame(records)
print(f'Generated {len(df):,} test records')
print(f'Date range: {df.test_date.min()} to {df.test_date.max()}')
print(df['result'].value_counts())
 


Generated 20,253 test records
Date range: 2024-07-01 to 2024-12-31
result
pass    17820
fail     2433
Name: count, dtype: int64


## Save as CSV

In [44]:
df.to_csv('data/roaming_tests.csv', index=False)
print('Data saved to data/roaming_tests.csv')

Data saved to data/roaming_tests.csv


## Save to MYSQL Database

In [46]:
# XAMAPP configurations

username= "root"
password= ""
host= "localhost"
database= "roaming_database"

*Create MYSQL engine*

In [47]:
engine= create_engine(f'mysql+pymysql://{username}:{password}@{host}/{database}')

#Save to MySQL database
df.to_sql('roaming_tests',con=engine, if_exists='replace',index=False)

20253

### Create partners dataframe

In [48]:
partners_df= pd.DataFrame(PARTNERS,columns=['partner_name','country','region','baseline_pass_rate'])

### SAve partners dataframe

In [49]:
partners_df.to_sql('partners', con=engine, if_exists='replace', index=False)

20

In [50]:
print('Saved: MySQL database "roaming_tests" with tables "roaming_tests" and "partners"')

Saved: MySQL database "roaming_tests" with tables "roaming_tests" and "partners"
